# Quaterback Support

Mojoolu Roberts

**Purpose**  
Evaluate quarterbacks on both individual performance and team context, highlighting who over- or under-performs relative to the support they receive.

**Intended Audience**  
- Analysts  
- Coaches / strategy discussions  
- Fans & media  

## Overview

Quarterbacks are graded along two dimensions:

1. **Support Grade (Team Context)** – Measures how much a quarterback benefits from his team. Components include pass protection, pass catching, play calling, run game, and defense & special teams.  
2. **Performance Grade (Individual Play)** – Evaluates the quarterback’s on-field efficiency, accuracy, and decision-making under pressure. Components include EPA per dropback, dropback success rate, completion percentage over expected, pressure-to-sack rate, and turnover-worthy plays.

A **dumbbell chart** visualizes the relationship between support and performance, making it easy to identify quarterbacks who excel despite poor support or benefit from strong team context.

## Grading Principles

- Weighted components clearly reflect importance  
- Percentile-based z-scores used to normalize metrics  
- Focus is on **performance relative to team support**, not absolute skill alone

In [1]:
# Required Libraries, uncomment the line below to install them if you haven't already
# %pip install numpy nflreadpy plotly polars pandas matplotlib scipy

In [2]:
import numpy as np
import nflreadpy as nfl
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from plotly import data
from pathlib import Path
import polars as pl
import pandas as pd
import json
import os
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy.special import erf

In [3]:
pio.renderers.default = "notebook"

os.makedirs("csv", exist_ok=True)

os.makedirs("figures/images/qbs", exist_ok=True)
os.makedirs("figures/html/qbs", exist_ok=True)

pio.templates["standard"] = dict(
    layout=dict(
        font=dict(size=14),
        paper_bgcolor="white",
        plot_bgcolor="#f5f5f5",

        xaxis=dict(
            showline=True,
            linecolor="black",
            gridcolor="#dcdcdc",
            zeroline=False,
            mirror=True
        ),
        yaxis=dict(
            showline=True,
            linecolor="black",
            gridcolor="#dcdcdc",
            zeroline=False,
            mirror=True
        ),

        shapes=[
            dict(
                type="rect",
                xref="paper",
                yref="paper",
                x0=0,
                y0=0,
                x1=1,
                y1=1,
                line=dict(color="black", width=2),
                fillcolor="rgba(0,0,0,0)"
            )
        ]
    )
)

pio.templates.default = "standard"

## Data Loading and Cleaning

In [4]:
pbp = nfl.load_pbp(seasons=[2025])
ftn = nfl.load_ftn_charting(seasons=[2025])
nextgen_passing = nfl.load_nextgen_stats(seasons=[2025], stat_type="passing")
nextgen_passing = nextgen_passing.filter(pl.col("season_type") == "REG").to_pandas()
nextgen_rushing = nfl.load_nextgen_stats(seasons=[2025], stat_type="rushing")
nextgen_rushing = nextgen_rushing.filter(pl.col("season_type") == "REG")
nextgen_receiving = nfl.load_nextgen_stats(seasons=[2025], stat_type="receiving")
nextgen_receiving = nextgen_receiving.filter(pl.col("season_type") == "REG")

teams = nfl.load_teams()
players = nfl.load_players().to_pandas()
teams = teams.select(["team_id", "team_abbr", "team_color", "team_logo_espn"])
teams_pd = teams.to_pandas()

teams_pd["team_id"] = teams_pd["team_id"].astype(str)

left_keys  = ["game_id", "play_id"]
right_keys = ["nflverse_game_id", "nflverse_play_id"]

pbp2 = pbp.with_columns([
    pl.col("play_id").cast(pl.Int64),
])

ftn2 = ftn.with_columns([
    pl.col("nflverse_play_id").cast(pl.Int64),
])

ftn_clean = ftn2.select(
    right_keys + [c for c in ftn2.columns if c not in pbp2.columns and c not in right_keys]
)

pbp = (
    pbp2
    .join(
        ftn_clean,
        left_on=left_keys,
        right_on=right_keys,
        how="left"
    )
)

pb = pbp.filter(pl.col("play_type") != "no_play")

In [5]:
with open("data/2025_ngs_team_passing_offense.json", "r") as f:
    data = json.load(f)

ngs_passing_offense = pd.json_normalize(data["offense"])

ngs_passing_offense_filtered = ngs_passing_offense[["teamId", "qbpPct", "ttp"]]

ngs_passing_offense_filtered = ngs_passing_offense_filtered.merge(
    teams_pd[["team_id", "team_abbr"]],
    left_on="teamId",
    right_on="team_id",
    how="left"
).drop(columns=["team_id"])

ngs_passing_offense_filtered = ngs_passing_offense_filtered.sort_values("qbpPct", ascending=False)

ngs_passing_offense_filtered = ngs_passing_offense_filtered.drop(ngs_passing_offense_filtered[ngs_passing_offense_filtered["team_abbr"].isin(["SD", "LAR", "STL", "OAK"])].index)

print(ngs_passing_offense_filtered.columns)
print(ngs_passing_offense_filtered.shape)
ngs_passing_offense_filtered.head(36)

Index(['teamId', 'qbpPct', 'ttp', 'team_abbr'], dtype='object')
(32, 4)


,teamId,qbpPct,ttp,team_abbr
34,1050,0.46479,2.73794,CLE
19,4400,0.42464,2.63321,LAC
35,3430,0.39869,2.55909,NYJ
8,3800,0.38688,2.76838,ARI
32,3000,0.38394,2.63768,MIN
5,3200,0.38336,2.84286,NE
23,3410,0.37778,2.90955,NYG
28,0750,0.37500,2.59227,NaN
33,2100,0.36615,2.86606,TEN
30,2520,0.36047,2.54778,LV


In [6]:
empty_team = ngs_passing_offense_filtered[ngs_passing_offense_filtered["team_abbr"].isna()]

empty_team

,teamId,qbpPct,ttp,team_abbr
28,0750,0.37500,2.59227,NaN
21,0200,0.35385,2.59104,NaN
29,0325,0.33659,2.84774,NaN
7,0920,0.33188,2.60045,NaN
11,0810,0.31318,2.87784,NaN
16,0610,0.28401,2.90069,NaN


In [7]:
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0750", ["team_abbr"]] = "CAR"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0200", ["team_abbr"]] = "ATL"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0325", ["team_abbr"]] = "BAL"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0920", ["team_abbr"]] = "CIN"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0810", ["team_abbr"]] = "CHI"
ngs_passing_offense_filtered.loc[ngs_passing_offense_filtered["teamId"] == "0610", ["team_abbr"]] = "BUF"

ngs_passing_offense_filtered.head(36)

,teamId,qbpPct,ttp,team_abbr
34,1050,0.46479,2.73794,CLE
19,4400,0.42464,2.63321,LAC
35,3430,0.39869,2.55909,NYJ
8,3800,0.38688,2.76838,ARI
32,3000,0.38394,2.63768,MIN
5,3200,0.38336,2.84286,NE
23,3410,0.37778,2.90955,NYG
28,0750,0.37500,2.59227,CAR
33,2100,0.36615,2.86606,TEN
30,2520,0.36047,2.54778,LV


In [8]:
dupes = ngs_passing_offense_filtered[ngs_passing_offense_filtered.duplicated(subset=["team_abbr"], keep=False)]

dupes

,teamId,qbpPct,ttp,team_abbr


In [9]:
ngs_qbs_filtered = nextgen_passing.loc[:, ["player_short_name", "player_gsis_id", "team_abbr", "attempts", "completion_percentage_above_expectation"]]

with open("data/2025_ngs_qb.json", "r") as f:
    data = json.load(f)

ngs_passers = pd.json_normalize(data["passers"])

ngs_qbs_filtered = (
    ngs_qbs_filtered
    .merge(
        players[["gsis_id","nfl_id", "headshot"]],
        left_on="player_gsis_id",
        right_on="gsis_id",
        how="left"
    )
    .merge(
        ngs_passers[["nflId", "qbp"]],
        left_on="nfl_id",
        right_on="nflId",
        how="left"
    )
    .drop(columns=["gsis_id", "nfl_id", "nflId"])
)

ngs_qbs_filtered = (
    ngs_qbs_filtered
    .sort_values("attempts", ascending=False)
    .head(35)
)

print(ngs_qbs_filtered.columns)
ngs_qbs_filtered.head()

Index(['player_short_name', 'player_gsis_id', 'team_abbr', 'attempts',
       'completion_percentage_above_expectation', 'headshot', 'qbp'],
      dtype='object')


,player_short_name,player_gsis_id,team_abbr,attempts,completion_percentage_above_expectation,headshot,qbp
29,B.Nix,00-0039732,DEN,612,-2.067418,https://static.www.nfl.com/image/upload/f_auto...,185
38,D.Prescott,00-0033077,DAL,600,4.445367,https://static.www.nfl.com/image/upload/f_auto...,211
1,M.Stafford,00-0026498,LAR,597,1.476147,https://static.www.nfl.com/image/upload/f_auto...,171
24,J.Goff,00-0033106,DET,578,1.581453,https://static.www.nfl.com/image/upload/f_auto...,212
0,C.Williams,00-0039918,CHI,568,-6.874683,https://static.www.nfl.com/image/upload/f_auto...,201


In [10]:
with open("data/2025_ngs_receiving.json", "r") as f:
    data = json.load(f)

ngs_receivers = pd.json_normalize(data["receivers"])

ngs_receivers_filtered = ngs_receivers.loc[:, ["teamId", "tgt", "xCatch", "rec"]]

ngs_receivers_filtered["xrec"] = (ngs_receivers_filtered["xCatch"] * ngs_receivers_filtered["tgt"]).astype(int)

ngs_receivers_filtered = ngs_receivers_filtered.drop(columns=["xCatch"])

print(ngs_receivers_filtered.shape)
ngs_receivers_filtered.head()

(565, 4)


,teamId,tgt,rec,xrec
0,3800,169,126,114
1,3800,126,78,72
2,0920,185,125,124
3,1400,124,74,69
4,1200,137,93,81


In [11]:
def grade_metric(df, col, high_is_good=True):
    mean = df[col].mean()
    std = df[col].std()

    z = (df[col] - mean) / std

    if not high_is_good:
        z = -z

    z = z.clip(-3, 3)

    grade = (0.5 * (1 + erf(z / np.sqrt(2)))) * 100

    return grade

In [ ]:
def make_table(df, grade_col, metric_cols, grade_cols, id_cols = None):
    pass

## Pass Protction (30%)

This metric evaluates the effectiveness of a quarterbacks pass protection in minimizing pressure situations and keeping the passer comfortable in the pocket. It combines the following components:

- **Non-QB-Fault Sacks** – Sacks allowed that are **not attributed to the quarterback**.  
- **Pressure Rate** – Rate of quarterback was pressures.  
- **Time to Pressure (TTP)** – Average time passed before quarterback was pressured.  

The overall pass protection score is calculated as the **weighted average of these three z-score percentiles**, contributing **30%** to the total QB support grade.

In [12]:
sack_not_qb = (
    pbp
    .filter(
        (pl.col("play_type") == "pass") &
        (pl.col("sack") == 1) &
        (pl.col("is_qb_fault_sack") == False)
    )
    .group_by("posteam")
    .agg(pl.count("play_id").alias("sack_not_qb"))
)
    
sack_not_qb = sack_not_qb.to_pandas()
sack_not_qb.head()

,posteam,sack_not_qb
0,TB,21
1,MIN,39
2,NE,44
3,JAX,30
4,PHI,24


In [13]:
pass_pro = (
    sack_not_qb
    .merge(
        ngs_passing_offense_filtered[["team_abbr", "qbpPct", "ttp"]],
        left_on="posteam",
        right_on="team_abbr",
        how="left"
    )
    .drop(columns=["team_abbr"])
)

print(pass_pro.shape)
pass_pro.head()

(32, 4)


,posteam,sack_not_qb,qbpPct,ttp
0,TB,21,0.31612,2.70571
1,MIN,39,0.38394,2.63768
2,NE,44,0.38336,2.84286
3,JAX,30,0.33282,2.69337
4,PHI,24,0.33043,2.77701


In [14]:
empty_sacks = pass_pro[pass_pro["sack_not_qb"].isna()]

empty_sacks

,posteam,sack_not_qb,qbpPct,ttp


In [15]:
pass_pro["pressure_rate_grade"] = grade_metric(pass_pro, "qbpPct", high_is_good=False)
pass_pro["time_to_pressure_grade"] = grade_metric(pass_pro, "ttp", high_is_good=True)
pass_pro["sack_grade"] = grade_metric(pass_pro, "sack_not_qb", high_is_good=False)

pass_pro["pass_pro_grade"] = (pass_pro["pressure_rate_grade"] + pass_pro["time_to_pressure_grade"] + pass_pro["sack_grade"]) / 3

pass_pro = pass_pro.sort_values("pass_pro_grade", ascending=False)

pass_pro.head()

,posteam,sack_not_qb,qbpPct,ttp,pressure_rate_grade,time_to_pressure_grade,sack_grade,pass_pro_grade
13,PIT,14,0.22559,2.81395,99.267577,75.872399,92.664013,89.267997
11,BUF,23,0.28401,2.90069,88.360296,90.947694,68.411561,82.573184
16,CHI,18,0.31318,2.87784,71.571711,87.903443,84.595218,81.356790
27,DEN,16,0.27720,2.74148,90.965664,56.804109,89.162587,78.977454
22,WAS,20,0.32753,2.86752,60.398313,86.315042,78.907943,75.207100


In [16]:
pass_pro_sheet = pass_pro.copy()

pass_pro_sheet["Sacks"] = pass_pro_sheet.apply(lambda row: f"{row['sack_not_qb']} ({row['sack_grade']:.1f})", axis=1)
pass_pro_sheet["Pres %"] = pass_pro_sheet.apply(lambda row: f"{(row['qbpPct'] * 100):.1f}% ({row['pressure_rate_grade']:.1f})", axis=1)
pass_pro_sheet["TTP"] = pass_pro_sheet.apply(lambda row: f"{row['ttp']:.2f} ({row['time_to_pressure_grade']:.1f})", axis=1)

pass_pro_sheet = pass_pro_sheet.rename(columns={"pass_pro_grade": "Grade", "posteam": "Team"})

pass_pro_sheet.drop(columns=["sack_not_qb", "qbpPct", "ttp", "pressure_rate_grade", "time_to_pressure_grade", "sack_grade"], inplace=True)
pass_pro_sheet["Grade"] = pass_pro_sheet["Grade"].apply(lambda x: f"{x:.1f}")
pass_pro_sheet.to_csv("csv/pass_pro.csv", index=False)
pass_pro_sheet.head()

,Team,Grade,Sacks,Pres %,TTP
13,PIT,89.3,14 (92.7),22.6% (99.3),2.81 (75.9)
11,BUF,82.6,23 (68.4),28.4% (88.4),2.90 (90.9)
16,CHI,81.4,18 (84.6),31.3% (71.6),2.88 (87.9)
27,DEN,79.0,16 (89.2),27.7% (91.0),2.74 (56.8)
22,WAS,75.2,20 (78.9),32.8% (60.4),2.87 (86.3)


## Play Calling (20% Weight)

This metric evaluates a quarterback’s playcaller in creating advantageous situations through their play design and pre-snap actions. It combines the following components:

- **Easy Button Rate** – The rate of plays designed to simplify execution, including screens, RPOs, or play-action passes.  
- **Motion Rate** – The frequency of pre-snap motion used to create mismatches or read defenses.  

The overall play calling score is calculated as the **average z-score percentile of these metrics**, contributing **20%** to the total QB support grade.

In [17]:
easy_button = (
    pbp
    .filter(pl.col("play_type").is_in(["run", "pass"]))
    .filter(pl.col("epa").is_not_null())
    .with_columns([
        (
            (
                (pl.col("is_rpo") == "true") |
                (pl.col("is_screen_pass") == "true") |
                (pl.col("is_play_action") == "true")
            )
        ).cast(pl.Int8).alias("easy_button_play")
    ])
    .group_by("posteam")
    .agg([
        pl.mean("easy_button_play").round(3).alias("easy_button_rate"),
    ])
    .sort("easy_button_rate", descending=True)
).to_pandas()

easy_button.head()

,posteam,easy_button_rate
0,NYG,0.272
1,KC,0.269
2,IND,0.267
3,DEN,0.266
4,TEN,0.266


In [18]:
motion = (
    pbp
    .filter(pl.col("play_type").is_in(["run", "pass"]))
    .filter(pl.col("epa").is_not_null())
    .with_columns([
        (pl.col("is_motion") == True).cast(pl.Int8).alias("motion_play")
    ])
    .group_by("posteam")
    .agg([
        pl.mean("motion_play").round(3).alias("motion_rate"),
    ])
    .sort("motion_rate", descending=True)
).to_pandas()

motion.head()

,posteam,motion_rate
0,MIA,0.706
1,SF,0.674
2,ATL,0.668
3,LA,0.635
4,NYJ,0.629


In [19]:
play_calling = (
    motion.merge(easy_button, on="posteam", how="left")
)

play_calling.head()

,posteam,motion_rate,easy_button_rate
0,MIA,0.706,0.249
1,SF,0.674,0.162
2,ATL,0.668,0.199
3,LA,0.635,0.214
4,NYJ,0.629,0.229


In [20]:
play_calling["easy_button_grade"] = grade_metric(play_calling, "easy_button_rate", high_is_good=True)
play_calling["motion_grade"] = grade_metric(play_calling, "motion_rate", high_is_good=True)

play_calling["play_calling_grade"] = (play_calling["easy_button_grade"] + play_calling["motion_grade"]) / 2

play_calling = play_calling.sort_values("play_calling_grade", ascending=False)

play_calling.head()

,posteam,motion_rate,easy_button_rate,easy_button_grade,motion_grade,play_calling_grade
0,MIA,0.706,0.249,76.032495,98.055581,87.044038
7,BUF,0.615,0.245,72.094594,80.174958,76.134776
11,CHI,0.587,0.256,82.129546,68.199265,75.164406
4,NYJ,0.629,0.229,53.938189,84.971027,69.454608
14,GB,0.550,0.264,87.772758,49.132740,68.452749


In [21]:
play_calling_sheet = play_calling.copy()

play_calling_sheet["Motion %"] = play_calling_sheet.apply(lambda row: f"{(row['motion_rate'] * 100):.1f}% ({row['motion_grade']:.1f})", axis=1)
play_calling_sheet["Easy Button %"] = play_calling_sheet.apply(lambda row: f"{(row['easy_button_rate'] * 100):.1f}% ({row['easy_button_grade']:.1f})", axis=1)

play_calling_sheet = play_calling_sheet.rename(columns={"play_calling_grade": "Grade", "posteam": "Team"})

play_calling_sheet.drop(columns=["motion_rate", "easy_button_rate", "easy_button_grade", "motion_grade"], inplace=True)
play_calling_sheet["Grade"] = play_calling_sheet["Grade"].apply(lambda x: f"{x:.1f}")
play_calling_sheet.to_csv("csv/play_calling.csv", index=False)
play_calling_sheet.head()

,Team,Grade,Motion %,Easy Button %
0,MIA,87.0,70.6% (98.1),24.9% (76.0)
7,BUF,76.1,61.5% (80.2),24.5% (72.1)
11,CHI,75.2,58.7% (68.2),25.6% (82.1)
4,NYJ,69.5,62.9% (85.0),22.9% (53.9)
14,GB,68.5,55.0% (49.1),26.4% (87.8)


## Pass Catching (30%)

This metric evaluates a quarterback's recivers ability to be reliable in the passing game by looking at how well their targets perform and how cleanly passes are caught. It combines the following components:

- **Average Separation** – The average distance between the receiver and the nearest defender at the time of the catch.  
- **Drop Rate** – The percentage of catchable passes that were dropped by the receiver.  
- **Catch Rate Over Expected (CROE)** – How often receivers catch passes relative to expectations based on difficulty.  
- **Average YAC (Yards After Catch) per Play** – Measures the additional yardage gained after the catch.  

The overall pass catching score is calculated as the **average z-score percentile across these metrics**, contributing **30%** to the total QB support grade.

In [22]:
seperation = (
    nextgen_receiving
    .group_by("team_abbr")
    .agg([
        pl.mean("avg_separation").alias("team_avg_separation"),
    ])
).to_pandas()

seperation.head()

,team_abbr,team_avg_separation
0,MIN,3.177674
1,JAX,2.895669
2,BUF,3.479611
3,ATL,3.083316
4,PHI,2.768429


In [23]:
drop_rate = (
    pbp
    .filter(pl.col("pass_attempt") == 1)
    .group_by("posteam")
    .agg(
        (
            pl.col("is_drop").sum() /
            pl.col("is_catchable_ball").sum()
        ).alias("drop_rate")
    )
).to_pandas()

drop_rate.head()

,posteam,drop_rate
0,LAC,0.047170
1,CIN,0.046809
2,DAL,0.058065
3,SF,0.056250
4,NYJ,0.086957


In [24]:
croe = (
    ngs_receivers_filtered
    .groupby("teamId")[["tgt","rec", "xrec"]]
    .sum()
    .reset_index()
)

croe["catch_rate"] = croe["rec"] / croe["tgt"]
croe["x_catch_rate"] = croe["xrec"] / croe["tgt"]
croe["catch_rate_over_expected"] = croe["catch_rate"] - croe["x_catch_rate"]

croe = croe.merge(
    teams_pd[["team_id", "team_abbr"]],
    left_on="teamId",
    right_on="team_id",
    how="left"
).drop(columns=["team_id"])

croe = croe.drop(croe[croe["team_abbr"].isin(["SD", "LAR", "STL", "OAK"])].index)

croe.loc[croe["teamId"] == "0750", ["team_abbr"]] = "CAR"
croe.loc[croe["teamId"] == "0200", ["team_abbr"]] = "ATL"
croe.loc[croe["teamId"] == "0325", ["team_abbr"]] = "BAL"
croe.loc[croe["teamId"] == "0920", ["team_abbr"]] = "CIN"
croe.loc[croe["teamId"] == "0810", ["team_abbr"]] = "CHI"
croe.loc[croe["teamId"] == "0610", ["team_abbr"]] = "BUF"

croe = croe.drop(columns=["teamId"])

croe.head()

,tgt,rec,xrec,catch_rate,x_catch_rate,catch_rate_over_expected,team_abbr
0,504,326,331,0.646825,0.656746,-0.009921,ATL
1,405,277,270,0.683951,0.666667,0.017284,BAL
2,504,363,342,0.720238,0.678571,0.041667,BUF
3,480,329,320,0.685417,0.666667,0.018750,CAR
4,532,334,366,0.627820,0.687970,-0.060150,CHI


In [25]:
yac = (
    pbp
    .group_by("posteam")
    .agg(pl.mean("yards_after_catch").alias("avg_yac"))
).to_pandas()

yac.head()

,posteam,avg_yac
0,None,NaN
1,CAR,5.011429
2,LV,5.311765
3,SEA,5.461942
4,TEN,4.785294


In [26]:
pass_catching = (
    drop_rate
    .merge(
        croe[["team_abbr", "catch_rate_over_expected"]], 
        left_on="posteam", 
        right_on="team_abbr",
        how="left")
    .drop(columns=["team_abbr"])
    .merge(
        seperation, 
        left_on="posteam", 
        right_on="team_abbr", 
        how="left")
    .drop(columns=["team_abbr"])
    .merge(
        yac, 
        on="posteam", 
        how="left"
    )
)

pass_catching.head()

,posteam,drop_rate,catch_rate_over_expected,team_avg_separation,avg_yac
0,LAC,0.047170,0.044280,2.751504,4.748705
1,CIN,0.046809,0.027961,2.789828,4.686893
2,DAL,0.058065,0.056013,2.969374,5.009547
3,SF,0.056250,0.057301,2.816968,4.633641
4,NYJ,0.086957,-0.022358,3.045830,4.439597


In [27]:
pass_catching["drop_rate_grade"] = grade_metric(pass_catching, "drop_rate", high_is_good=False)
pass_catching["catch_rate_over_expected_grade"] = grade_metric(pass_catching, "catch_rate_over_expected", high_is_good=True)
pass_catching["separation_grade"] = grade_metric(pass_catching, "team_avg_separation", high_is_good=True)
pass_catching["yac_grade"] = grade_metric(pass_catching, "avg_yac", high_is_good=True)

pass_catching["pass_catching_grade"] = (
    pass_catching[["drop_rate_grade", "catch_rate_over_expected_grade", "separation_grade", "yac_grade"]]
    .mean(axis=1)
)

pass_catching = pass_catching.sort_values("pass_catching_grade", ascending=False)

pass_catching.head()

,posteam,drop_rate,catch_rate_over_expected,team_avg_separation,avg_yac,drop_rate_grade,catch_rate_over_expected_grade,separation_grade,yac_grade,pass_catching_grade
9,SEA,0.039409,0.057471,3.174221,5.461942,91.301243,87.861249,74.491547,73.346675,81.750178
23,BUF,0.063348,0.041667,3.479611,5.788413,37.116874,76.066530,97.519859,89.694232,75.099374
17,DET,0.056471,0.029091,3.199058,6.119289,56.209738,63.409245,77.777209,97.218716,73.653727
7,MIA,0.052023,0.021692,3.207940,5.590062,68.080102,55.074104,78.890923,80.918048,70.740795
21,NE,0.041304,0.112266,2.976531,5.109049,88.987532,99.712441,42.619423,47.228235,69.636908


In [28]:
pass_catching_sheet = pass_catching.copy()

pass_catching_sheet["Drop %"] = pass_catching_sheet.apply(lambda row: f"{(row['drop_rate'] * 100):.1f}% ({row['drop_rate_grade']:.1f})", axis=1)
pass_catching_sheet["CROE"] = pass_catching_sheet.apply(lambda row: f"{(row['catch_rate_over_expected'] * 100):.1f}% ({row['catch_rate_over_expected_grade']:.1f})", axis=1)
pass_catching_sheet["Sep"] = pass_catching_sheet.apply(lambda row: f"{row['team_avg_separation']:.1f} ({row['separation_grade']:.1f})", axis=1)
pass_catching_sheet["YAC"] = pass_catching_sheet.apply(lambda row: f"{row['avg_yac']:.1f} ({row['yac_grade']:.1f})", axis=1)

pass_catching_sheet = pass_catching_sheet.rename(columns={"pass_catching_grade": "Grade", "posteam": "Team"})

pass_catching_sheet.drop(columns=["drop_rate", "drop_rate_grade", "catch_rate_over_expected", "catch_rate_over_expected_grade", "team_avg_separation", "separation_grade", "avg_yac", "yac_grade"], inplace=True)
pass_catching_sheet["Grade"] = pass_catching_sheet["Grade"].apply(lambda x: f"{x:.1f}")
pass_catching_sheet.to_csv("csv/pass_catching.csv", index=False)
pass_catching_sheet.head()

,Team,Grade,Drop %,CROE,Sep,YAC
9,SEA,81.8,3.9% (91.3),5.7% (87.9),3.2 (74.5),5.5 (73.3)
23,BUF,75.1,6.3% (37.1),4.2% (76.1),3.5 (97.5),5.8 (89.7)
17,DET,73.7,5.6% (56.2),2.9% (63.4),3.2 (77.8),6.1 (97.2)
7,MIA,70.7,5.2% (68.1),2.2% (55.1),3.2 (78.9),5.6 (80.9)
21,NE,69.6,4.1% (89.0),11.2% (99.7),3.0 (42.6),5.1 (47.2)


## Run Game (15%)

This metric evaluates a quarterback's support from their running game and how effectively the offense gains ground on runs plays. It combines the following components:

- **EPA per Rush** – Average expected points added per rushing play.  
- **Run Success Rate** – The percentage of rushing plays that meet success criteria (epa > 0).  
- **Average Yards per Carry Over Expected (YPCOE)** – Measures how many yards above or below expectation running backs gain on runs.  

The overall run game score is calculated as the **average z-score percentile across these metrics**, contributing **15%** to the total QB support grade.

In [29]:
rush_sr = (
    pbp
    .filter(
        (pl.col("play_type") == "run") &
        (pl.col("qb_scramble") == 0) 
        )
    .group_by("posteam")
    .agg(
        (pl.mean("success")).alias("rush_sr")
    )
).to_pandas()

rush_sr.head()

,posteam,rush_sr
0,GB,0.438178
1,CHI,0.458984
2,BUF,0.480000
3,LV,0.312865
4,MIN,0.437500


In [30]:
rush_epa = (
    pbp
    .filter(
        (pl.col("play_type") == "run") &
        (pl.col("qb_scramble") == 0) 
        )
    .group_by("posteam")
    .agg(
        (pl.mean("epa")).alias("rush_epa")
    )
).to_pandas()

rush_epa.head()

,posteam,rush_epa
0,WAS,-0.036947
1,SF,-0.043213
2,ARI,-0.112721
3,MIA,-0.027657
4,NYJ,-0.103231


In [31]:
rush_ypcoe = (
    nextgen_rushing
    .group_by("team_abbr")
    .agg([pl.mean("rush_yards_over_expected_per_att").alias("rush_ypcoe")])
).to_pandas()

rush_ypcoe.loc[rush_ypcoe["team_abbr"] == "LAR", ["team_abbr"]] = "LA"

rush_ypcoe.head()

,team_abbr,rush_ypcoe
0,WAS,0.830207
1,NYJ,0.453602
2,CAR,0.099533
3,SEA,0.342880
4,SF,-0.534057


In [32]:
run_game = (
    rush_sr
    .merge(rush_epa, on="posteam", how="left")
    .merge(rush_ypcoe, left_on="posteam", right_on="team_abbr", how="left")
    .drop(columns=["team_abbr"])
)

run_game.head()

,posteam,rush_sr,rush_epa,rush_ypcoe
0,GB,0.438178,-0.066720,-0.016411
1,CHI,0.458984,0.002676,0.428280
2,BUF,0.480000,0.069211,1.181998
3,LV,0.312865,-0.303572,-0.280740
4,MIN,0.437500,-0.054451,0.353924


In [33]:
run_game["rush_sr_grade"] = grade_metric(run_game, "rush_sr", high_is_good=True)
run_game["rush_epa_grade"] = grade_metric(run_game, "rush_epa", high_is_good=True)
run_game["rush_ypcoe_grade"] = grade_metric(run_game, "rush_ypcoe", high_is_good=True)

run_game["run_game_grade"] = (
    run_game[["rush_sr_grade", "rush_epa_grade", "rush_ypcoe_grade"]]
    .mean(axis=1)
)

run_game = run_game.sort_values("run_game_grade", ascending=False)

run_game.head()

,posteam,rush_sr,rush_epa,rush_ypcoe,rush_sr_grade,rush_epa_grade,rush_ypcoe_grade,run_game_grade
2,BUF,0.480000,0.069211,1.181998,96.451405,95.807343,97.095436,96.451395
21,LA,0.496078,0.036411,0.697312,98.748180,89.593904,78.484042,88.942042
16,BAL,0.423841,0.051947,1.170848,61.242420,93.074590,96.922735,83.746581
31,IND,0.450739,0.066621,0.513206,84.462944,95.464224,64.371262,81.432810
6,PIT,0.442500,-0.001969,0.875692,78.543208,76.076544,88.411703,81.010485


In [34]:
run_game_sheet = run_game.copy()

run_game_sheet["SR"] = run_game_sheet.apply(lambda row: f"{(row['rush_sr'] * 100):.1f}% ({row['rush_sr_grade']:.1f})", axis=1)
run_game_sheet["EPA"] = run_game_sheet.apply(lambda row: f"{row['rush_epa']:.2f} ({row['rush_epa_grade']:.1f})", axis=1)
run_game_sheet["YPCOE"] = run_game_sheet.apply(lambda row: f"{row['rush_ypcoe']:.2f} ({row['rush_ypcoe_grade']:.1f})", axis=1)

run_game_sheet = run_game_sheet.rename(columns={"run_game_grade": "Grade", "posteam": "Team"})

run_game_sheet.drop(columns=["rush_sr", "rush_sr_grade", "rush_epa", "rush_epa_grade", "rush_ypcoe", "rush_ypcoe_grade"], inplace=True)
run_game_sheet["Grade"] = run_game_sheet["Grade"].apply(lambda x: f"{x:.1f}")
run_game_sheet.to_csv("csv/run_game.csv", index=False)
run_game_sheet.head()

,Team,Grade,SR,EPA,YPCOE
2,BUF,96.5,48.0% (96.5),0.07 (95.8),1.18 (97.1)
21,LA,88.9,49.6% (98.7),0.04 (89.6),0.70 (78.5)
16,BAL,83.7,42.4% (61.2),0.05 (93.1),1.17 (96.9)
31,IND,81.4,45.1% (84.5),0.07 (95.5),0.51 (64.4)
6,PIT,81.0,44.2% (78.5),-0.00 (76.1),0.88 (88.4)


## Defense and Special Teams (5%)

This metric captures the quarterback's indirect support from their defense and special teams. It combines the following components:

- **Defensive EPA per Play** – Average expected points allowed per defensive play.  
- **Total Special Teams EPA** – Contribution of special teams to field position and scoring opportunities.  
- **Average Starting Field Position** – How advantageous the offense’s starting field position is after defensive or special teams plays.  

The overall defense & special teams score is calculated as the **average z-score percentile across these metrics**, contributing **5%** to the total QB support grade.

In [35]:
def_epa = (
    pbp
    .filter(pl.col("play_type").is_in(["run", "pass"]))
    .group_by("defteam")
    .agg(pl.mean("epa").alias("def_epa"))
).to_pandas()

def_epa.head()

,defteam,def_epa
0,NO,-0.059060
1,BUF,-0.010997
2,CHI,0.031369
3,NYG,0.087394
4,WAS,0.142487


In [36]:
special_teams_epa_off = (
    pbp
    .filter(pl.col("special") == 1)
    .group_by("posteam")
    .agg(pl.sum("epa").alias("special_teams_epa_off"))
)

special_teams_epa_def = (
    pbp
    .filter(pl.col("special") == 1)
    .group_by("defteam")
    .agg(-pl.sum("epa").alias("special_teams_epa_def"))
)

special_teams_epa = (
    special_teams_epa_off
    .join(
        special_teams_epa_def,
        left_on="posteam",
        right_on="defteam",
        how="inner"
    )
).to_pandas()

special_teams_epa["special_teams_epa"] = special_teams_epa["special_teams_epa_off"] + special_teams_epa["special_teams_epa_def"]

special_teams_epa = special_teams_epa.drop(columns=["special_teams_epa_off", "special_teams_epa_def"])

special_teams_epa.head()

,posteam,special_teams_epa
0,LA,-33.860137
1,NYG,-7.586324
2,NE,-5.620092
3,WAS,25.965711
4,TEN,21.799807


In [37]:
avg_start = (
    pbp
    .filter(
        pl.col("play_type").is_in(["run", "pass"]) &
        (pl.col("play_id") == pl.col("drive_play_id_started"))
    )
    .group_by("posteam")
    .agg(pl.mean("yardline_100").alias("avg_start"))
).to_pandas()

avg_start.head()

,posteam,avg_start
0,KC,69.573171
1,TB,67.976471
2,IND,69.948052
3,NO,70.425000
4,CAR,70.448718


In [38]:
def_and_st = (
    def_epa
    .merge(
        special_teams_epa[["posteam", "special_teams_epa"]],
        left_on="defteam",
        right_on="posteam",
        how="left"
    )
    .drop(columns=["posteam"])
    .merge(
        avg_start,
        left_on="defteam",
        right_on="posteam",
        how="left"
    )
    .drop(columns=["posteam"])
)

def_and_st.head()

,defteam,def_epa,special_teams_epa,avg_start
0,NO,-0.059060,-65.265423,70.425000
1,BUF,-0.010997,-33.418414,71.229167
2,CHI,0.031369,16.810705,66.636364
3,NYG,0.087394,-7.586324,69.520548
4,WAS,0.142487,25.965711,69.393443


In [39]:
def_and_st["def_epa_grade"] = grade_metric(def_and_st, "def_epa", high_is_good=False)
def_and_st["special_teams_epa_grade"] = grade_metric(def_and_st, "special_teams_epa", high_is_good=True)
def_and_st["avg_start_grade"] = grade_metric(def_and_st, "avg_start", high_is_good=False)

def_and_st["def_and_st_grade"] = (
    def_and_st[["def_epa_grade", "special_teams_epa_grade", "avg_start_grade"]]
    .mean(axis=1)
)

def_and_st = def_and_st.sort_values("def_and_st_grade", ascending=False)

def_and_st.head()

,defteam,def_epa,special_teams_epa,avg_start,def_epa_grade,special_teams_epa_grade,avg_start_grade,def_and_st_grade
23,HOU,-0.160723,51.777008,66.520000,97.334576,95.076341,96.797166,96.402694
8,JAX,-0.086168,26.744811,66.914286,85.806162,80.330257,94.740986,86.959135
7,SEA,-0.123929,72.363005,69.314961,93.417323,98.953483,58.334296,83.568367
16,MIN,-0.099459,13.628391,69.067416,88.973051,66.818520,63.901346,73.230972
2,CHI,0.031369,16.810705,66.636364,38.768722,70.418002,96.274354,68.487026


In [40]:
def_and_st_sheet = def_and_st.copy()

def_and_st_sheet["Def EPA"] = def_and_st_sheet.apply(lambda row: f"{row['def_epa']:.2f} ({row['def_epa_grade']:.1f})", axis=1)
def_and_st_sheet["ST EPA"] = def_and_st_sheet.apply(lambda row: f"{row['special_teams_epa']:.2f} ({row['special_teams_epa_grade']:.1f})", axis=1)
def_and_st_sheet["Start"] = def_and_st_sheet.apply(
    lambda row: (
        f"{'OWN' if row['avg_start'] > 50 else 'OPP'} "
        f"{(row['avg_start'] if row['avg_start'] <= 50 else 100 - row['avg_start']):.1f} "
        f"({row['avg_start_grade']:.1f})"
    ),
    axis=1
)

def_and_st_sheet = def_and_st_sheet.rename(columns={"def_and_st_grade": "Grade", "defteam": "Team"})
def_and_st_sheet.drop(columns=["def_epa", "def_epa_grade", "special_teams_epa", "special_teams_epa_grade", "avg_start", "avg_start_grade"], inplace=True)
def_and_st_sheet["Grade"] = def_and_st_sheet["Grade"].apply(lambda x: f"{x:.1f}")
def_and_st_sheet.to_csv("csv/def_and_st_sheet.csv", index=False)
def_and_st_sheet.head()

,Team,Grade,Def EPA,ST EPA,Start
23,HOU,96.4,-0.16 (97.3),51.78 (95.1),OWN 33.5 (96.8)
8,JAX,87.0,-0.09 (85.8),26.74 (80.3),OWN 33.1 (94.7)
7,SEA,83.6,-0.12 (93.4),72.36 (99.0),OWN 30.7 (58.3)
16,MIN,73.2,-0.10 (89.0),13.63 (66.8),OWN 30.9 (63.9)
2,CHI,68.5,0.03 (38.8),16.81 (70.4),OWN 33.4 (96.3)


## Quarterback Support Grade

The Quarterback Support Grade measures how much a quarterback’s team helps him succeed, combining contributions from the offensive line, receivers, play design, running game, and overall team context. The components are:

- **Pass Protection (30%)** – Evaluates how well the offensive line protects the quarterback, factoring in non-QB-fault sacks, pressure rate, and time to pressure.  
- **Pass Catching (30%)** – Measures the effectiveness of receivers and passing plays, including average separation, drop rate, catch rate over expected, and yards after catch per play.  
- **Play Calling (20%)** – Assesses the quarterback’s rate of advantageous play designs and pre-snap actions, including easy button rate (screens, RPOs, play action) and motion rate.  
- **Run Game (15%)** – Evaluates contribution from the running game via EPA per rush, run success rate, and yards per carry over expected.  
- **Defense & Special Teams (5%)** – Captures the indirect support from team defense and special teams, including defensive EPA per play, total special teams EPA, and average starting field position.

In [41]:
qbs = (
    pass_pro[["posteam", "pass_pro_grade"]]
    .merge(
        play_calling[["posteam", "play_calling_grade"]],
        on="posteam",
        how="left"
    )
    .merge(
        pass_catching[["posteam", "pass_catching_grade"]],
        on="posteam",
        how="left"
    )
    .merge(
        run_game[["posteam", "run_game_grade"]],
        on="posteam",
        how="left"
    )
    .merge(
        def_and_st[["defteam", "def_and_st_grade"]],
        left_on="posteam",
        right_on="defteam",
        how="left"
    )
    .drop(columns=["defteam"])
)

qbs.head()

,posteam,pass_pro_grade,play_calling_grade,pass_catching_grade,run_game_grade,def_and_st_grade
0,PIT,89.267997,67.814114,66.861899,81.010485,63.530372
1,BUF,82.573184,76.134776,75.099374,96.451395,30.143135
2,CHI,81.356790,75.164406,45.802606,74.737791,68.487026
3,DEN,78.977454,56.308891,49.299302,50.028282,52.828610
4,WAS,75.207100,63.482212,37.811462,70.059505,47.334286


In [42]:
qbs["quarterback_support_grade"] = (
    (0.3 * qbs["pass_pro_grade"]) +
    (0.2 * qbs["play_calling_grade"]) +
    (0.3 * qbs["pass_catching_grade"]) +
    (0.15 * qbs["run_game_grade"]) +
    (0.05 * qbs["def_and_st_grade"])
)

qbs = qbs.sort_values("quarterback_support_grade", ascending=False)

col = qbs.pop("quarterback_support_grade")
qbs.insert(1, "quarterback_support_grade", col)

qbs.head()

,posteam,quarterback_support_grade,pass_pro_grade,play_calling_grade,pass_catching_grade,run_game_grade,def_and_st_grade
1,BUF,78.503588,82.573184,76.134776,75.099374,96.451395,30.143135
0,PIT,75.729883,89.267997,67.814114,66.861899,81.010485,63.530372
2,CHI,67.815720,81.356790,75.164406,45.802606,74.737791,68.487026
5,SEA,63.815128,72.215685,36.479730,81.750178,41.006697,83.568367
9,LA,62.381906,63.545474,61.402086,50.284073,88.942042,52.226363


In [43]:
qbs_sheet = qbs.copy()

qbs_sheet = qbs_sheet.rename(columns={
    "posteam": "Team",
    "quarterback_support_grade": "QBS Grade",
    "pass_pro_grade": "Pass Pro",
    "play_calling_grade": "Play Call",
    "pass_catching_grade": "Pass Catching",
    "run_game_grade": "Run Game",
    "def_and_st_grade": "Def/ST"
})

qbs_sheet = qbs_sheet.applymap(lambda x: f"{x:.1f}" if isinstance(x, float) else x)
qbs_sheet.to_csv("csv/qb_support.csv", index=False)
qbs_sheet.head()

/var/folders/_v/f9jlhvnd2xg9kybwfc9yh6nw0000gn/T/ipykernel_34934/3817257252.py:13: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



,Team,QBS Grade,Pass Pro,Play Call,Pass Catching,Run Game,Def/ST
1,BUF,78.5,82.6,76.1,75.1,96.5,30.1
0,PIT,75.7,89.3,67.8,66.9,81.0,63.5
2,CHI,67.8,81.4,75.2,45.8,74.7,68.5
5,SEA,63.8,72.2,36.5,81.8,41.0,83.6
9,LA,62.4,63.5,61.4,50.3,88.9,52.2


## Quaeterback Performance Grade

This metric evaluates the quarterback's direct on-field performance, combining efficiency, accuracy, and decision-making under pressure. The components and weights are:

- **EPA per Dropback (35%)** – Expected points added per dropback, measuring overall offensive impact.  
- **Dropback Success Rate (30%)** – Percentage of dropbacks considered successful (epa > 0).  
- **Completion Percentage Over Expected (CPOE) (25%)** – How often the quarterback completes passes relative to expected difficulty.  
- **Pressure-to-Sack Rate (5%)** – The rate of plays under pressure that result in a sack **attributed to the quarterback**.  
- **Turnover-Worthy Play Rate (5%)** – The proportion of dropbacks resulting in fumbles or interception-worthy throws.  

The overall quarterback performance grade is calculated as the **weighted average z-score percentile of these metrics**.

In [44]:
qb_pbp = (
    pbp
    .filter(pl.col("qb_dropback") == 1)
    .with_columns([
        pl.col("is_qb_fault_sack").cast(pl.Int8).alias("qb_fault_sack"),

        (pl.col("passer_player_id") == pl.col("fumbled_1_player_id"))
        .fill_null(False)
        .cast(pl.Int8)
        .alias("qb_fumble"),

        (
            pl.col("is_interception_worthy").fill_null(False) |
            (pl.col("passer_player_id") == pl.col("fumbled_1_player_id")).fill_null(False)
        )
        .cast(pl.Int8)
        .alias("turnover_worthy")
    ])
    .group_by("passer_player_id")
    .agg([
        pl.len().alias("dropbacks"),
        pl.mean("epa").alias("epa_per_dropback"),
        pl.mean("success").alias("success_rate"),
        pl.sum("qb_fault_sack").alias("qb_fault_sacks"),
        pl.mean("turnover_worthy").alias("turnover_worthy_play_rate"),
    ])
).to_pandas()

qb_pbp.head()

,passer_player_id,dropbacks,epa_per_dropback,success_rate,qb_fault_sacks,turnover_worthy_play_rate
0,00-0036972,308,0.116467,0.516234,1,0.045455
1,00-0026498,741,0.221393,0.518219,10,0.036437
2,00-0039164,2,-0.605810,0.500000,0,0.000000
3,00-0040730,1,-0.458608,0.000000,0,1.000000
4,00-0035100,134,-0.394705,0.380597,0,0.074627


In [45]:
qb_overall = (
    ngs_qbs_filtered
    .merge(
        qb_pbp,
        left_on="player_gsis_id",
        right_on="passer_player_id",
        how="left"
    )
    .drop(columns=["passer_player_id", "player_gsis_id"])
)

qb_overall["qb_pressure_to_sack_rate"] = qb_overall["qb_fault_sacks"] / qb_overall["qbp"]

qb_overall = qb_overall.drop(columns=["qb_fault_sacks", "qbp"])
qb_overall = qb_overall.sort_values("qb_pressure_to_sack_rate")
qb_overall.head()

,player_short_name,team_abbr,attempts,completion_percentage_above_expectation,headshot,dropbacks,epa_per_dropback,success_rate,turnover_worthy_play_rate,qb_pressure_to_sack_rate
26,M.Jones,SF,289,2.969585,https://static.www.nfl.com/image/upload/f_auto...,308,0.116467,0.516234,0.045455,0.011111
21,T.Tagovailoa,MIA,384,-0.317370,https://static.www.nfl.com/image/upload/f_auto...,415,0.019177,0.448193,0.055422,0.024194
31,S.Rattler,NO,257,3.048521,https://static.www.nfl.com/image/upload/f_auto...,272,-0.131853,0.441176,0.051471,0.026087
18,J.Love,GB,439,3.816196,https://static.www.nfl.com/image/upload/f_auto...,508,0.253294,0.492126,0.029528,0.028902
30,J.Burrow,CIN,259,2.055212,https://static.www.nfl.com/image/upload/f_auto...,278,0.099146,0.489209,0.014388,0.032967


In [46]:
qb_overall["epa_per_dropback_grade"] = grade_metric(qb_overall, "epa_per_dropback", high_is_good=True)
qb_overall["success_rate_grade"] = grade_metric(qb_overall, "success_rate", high_is_good=True)
qb_overall["cpoe_grade"] = grade_metric(qb_overall, "completion_percentage_above_expectation", high_is_good=True)
qb_overall["qb_sack_grade"] = grade_metric(qb_overall, "qb_pressure_to_sack_rate", high_is_good=False)
qb_overall["turnover_worthy_grade"] = grade_metric(qb_overall, "turnover_worthy_play_rate", high_is_good=False)

qb_overall.head()

,player_short_name,team_abbr,attempts,completion_percentage_above_expectation,headshot,dropbacks,epa_per_dropback,success_rate,turnover_worthy_play_rate,qb_pressure_to_sack_rate,epa_per_dropback_grade,success_rate_grade,cpoe_grade,qb_sack_grade,turnover_worthy_grade
26,M.Jones,SF,289,2.969585,https://static.www.nfl.com/image/upload/f_auto...,308,0.116467,0.516234,0.045455,0.011111,76.301873,93.811124,76.695716,94.440753,22.572145
21,T.Tagovailoa,MIA,384,-0.317370,https://static.www.nfl.com/image/upload/f_auto...,415,0.019177,0.448193,0.055422,0.024194,47.577487,45.843242,40.036085,88.912833,4.349402
31,S.Rattler,NO,257,3.048521,https://static.www.nfl.com/image/upload/f_auto...,272,-0.131853,0.441176,0.051471,0.026087,10.264007,39.209662,77.410317,87.863965,9.150471
18,J.Love,GB,439,3.816196,https://static.www.nfl.com/image/upload/f_auto...,508,0.253294,0.492126,0.029528,0.028902,96.473767,83.066570,83.685324,86.178930,78.189559
30,J.Burrow,CIN,259,2.055212,https://static.www.nfl.com/image/upload/f_auto...,278,0.099146,0.489209,0.014388,0.032967,71.828090,81.228073,67.576514,83.475123,98.727371


In [47]:
qbp = qb_overall.loc[:, ["player_short_name", "team_abbr", "headshot", "attempts", "epa_per_dropback_grade", "success_rate_grade", "cpoe_grade", "qb_sack_grade", "turnover_worthy_grade"]]

qbp["quarterback_performance_grade"] = (
    (0.35 * qbp["epa_per_dropback_grade"]) +
    (0.3 * qbp["success_rate_grade"]) +
    (0.25 * qbp["cpoe_grade"]) +
    (0.05 * qbp["qb_sack_grade"]) +
    (0.05 * qbp["turnover_worthy_grade"])
)

qbp = qbp.sort_values("quarterback_performance_grade", ascending=False)

qbp.loc[qbp["team_abbr"] == "LAR", ["team_abbr"]] = "LA"

qbp.head()

,player_short_name,team_abbr,headshot,attempts,epa_per_dropback_grade,success_rate_grade,cpoe_grade,qb_sack_grade,turnover_worthy_grade,quarterback_performance_grade
18,J.Love,GB,https://static.www.nfl.com/image/upload/f_auto...,439,96.473767,83.066570,83.685324,86.178930,78.189559,87.825545
11,D.Maye,NE,https://static.www.nfl.com/image/upload/f_auto...,492,88.190799,89.815946,99.492261,12.552888,36.851277,85.154837
27,B.Purdy,SF,https://static.www.nfl.com/image/upload/f_auto...,284,78.829533,93.537249,91.252355,80.712020,7.059286,82.853165
2,M.Stafford,LA,https://static.www.nfl.com/image/upload/f_auto...,597,93.988631,94.374966,61.141814,59.856853,54.543627,82.213988
14,S.Darnold,SEA,https://static.www.nfl.com/image/upload/f_auto...,477,82.704372,93.185518,87.034645,18.661024,28.911171,81.039457


In [48]:
qbp_sheet = qbp.copy()

qbp_sheet = qbp_sheet.merge(
    qb_overall[["player_short_name", "epa_per_dropback", "success_rate", "completion_percentage_above_expectation", "qb_pressure_to_sack_rate", "turnover_worthy_play_rate"]],
    on="player_short_name",
    how="left"
)

qbp_sheet["EPA"] = qbp_sheet.apply(lambda row: f"{(row['epa_per_dropback']):.2f} ({row['epa_per_dropback_grade']:.1f})", axis=1)
qbp_sheet["SR"] = qbp_sheet.apply(lambda row: f"{(row['success_rate'] * 100):.1f}% ({row['success_rate_grade']:.1f})", axis=1)
qbp_sheet["CPOE"] = qbp_sheet.apply(lambda row: f"{(row['completion_percentage_above_expectation']):.2f} ({row['cpoe_grade']:.1f})", axis=1)
qbp_sheet["Pressure to Sack Rate"] = qbp_sheet.apply(lambda row: f"{(row['qb_pressure_to_sack_rate'] * 100):.1f}% ({row['qb_sack_grade']:.1f})", axis=1)
qbp_sheet["Turnover Worthy Play Rate"] = qbp_sheet.apply(lambda row: f"{(row['turnover_worthy_play_rate'] * 100):.1f}% ({row['turnover_worthy_grade']:.1f})", axis=1)

qbp_sheet = qbp_sheet.rename(columns={"player_short_name": "Passer", "team_abbr": "Team", "quarterback_performance_grade": "Grade"})

qbp_sheet["Grade"] = qbp_sheet["Grade"].apply(lambda x: f"{x:.1f}")

qbp_sheet = qbp_sheet[["Passer", "Team", "Grade", "EPA", "SR", "CPOE", "Pressure to Sack Rate", "Turnover Worthy Play Rate"]]
qbp_sheet.to_csv("csv/qb_performance.csv", index=False)
qbp_sheet.head()

,Passer,Team,Grade,EPA,SR,CPOE,Pressure to Sack Rate,Turnover Worthy Play Rate
0,J.Love,GB,87.8,0.25 (96.5),49.2% (83.1),3.82 (83.7),2.9% (86.2),3.0% (78.2)
1,D.Maye,NE,85.2,0.18 (88.2),50.5% (89.8),9.14 (99.5),10.8% (12.6),4.1% (36.9)
2,B.Purdy,SF,82.9,0.13 (78.8),51.5% (93.5),5.07 (91.3),3.7% (80.7),5.3% (7.1)
3,M.Stafford,LA,82.2,0.22 (94.0),51.8% (94.4),1.48 (61.1),5.8% (59.9),3.6% (54.5)
4,S.Darnold,SEA,81.0,0.14 (82.7),51.4% (93.2),4.31 (87.0),9.9% (18.7),4.3% (28.9)


## Quarterback Performance vs Support (Top 25 in pass attempts)

This dumbbell chart visualizes how each quarterback’s **on-field performance** compares to the **support they received** from their team.  

- **X-Axis** –  Quarterback names.
- **Y-Axis** – Scores or percentiles for quarterbacks. 
- **Connecting Line** – Highlights the gap between support and performance for each quarterback, making it easy to see who over- or under-performed relative to their team.  

This chart is useful for identifying quarterbacks who excel despite poor support or those who benefit from strong team context.

In [49]:
support_and_performance = (
    qbp[["player_short_name", "team_abbr", "headshot", "quarterback_performance_grade", "attempts"]]
    .merge(
        qbs[["posteam", "quarterback_support_grade"]],
        left_on=["team_abbr"],
        right_on=["posteam"],
        how="left"
    )
    .merge(
        teams_pd[["team_abbr", "team_color", "team_logo_espn"]],
        left_on="team_abbr",
        right_on="team_abbr",
        how="left"
    )
    .drop(columns=["posteam"])
)

support_and_performance.head()

,player_short_name,team_abbr,headshot,quarterback_performance_grade,attempts,quarterback_support_grade,team_color,team_logo_espn
0,J.Love,GB,https://static.www.nfl.com/image/upload/f_auto...,87.825545,439,56.596926,#203731,https://a.espncdn.com/i/teamlogos/nfl/500/gb.png
1,D.Maye,NE,https://static.www.nfl.com/image/upload/f_auto...,85.154837,492,44.139139,#002244,https://a.espncdn.com/i/teamlogos/nfl/500/ne.png
2,B.Purdy,SF,https://static.www.nfl.com/image/upload/f_auto...,82.853165,284,50.185485,#AA0000,https://a.espncdn.com/i/teamlogos/nfl/500/sf.png
3,M.Stafford,LA,https://static.www.nfl.com/image/upload/f_auto...,82.213988,597,62.381906,#003594,https://a.espncdn.com/i/teamlogos/nfl/500/lar.png
4,S.Darnold,SEA,https://static.www.nfl.com/image/upload/f_auto...,81.039457,477,63.815128,#002244,https://a.espncdn.com/i/teamlogos/nfl/500/sea.png


In [50]:
support_and_performance_sheet = support_and_performance.copy()

support_and_performance_sheet = support_and_performance_sheet.rename(columns={
    "player_short_name": "Passer",
    "team_abbr": "Team",
    "quarterback_performance_grade": "Performance Grade",
    "quarterback_support_grade": "Support Grade",
})

support_and_performance_sheet = support_and_performance_sheet.drop(columns=["headshot", "attempts", "team_logo_espn", "team_color"])
support_and_performance_sheet = support_and_performance_sheet.applymap(lambda x: f"{x:.1f}" if isinstance(x, float) else x)
support_and_performance_sheet.to_csv("csv/support_and_performance.csv", index=False)
support_and_performance_sheet.head()

/var/folders/_v/f9jlhvnd2xg9kybwfc9yh6nw0000gn/T/ipykernel_34934/420469285.py:11: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



,Passer,Team,Performance Grade,Support Grade
0,J.Love,GB,87.8,56.6
1,D.Maye,NE,85.2,44.1
2,B.Purdy,SF,82.9,50.2
3,M.Stafford,LA,82.2,62.4
4,S.Darnold,SEA,81.0,63.8


In [54]:
pdf = (
    support_and_performance
    .sort_values("attempts", ascending=False)
    .head(25)
    .sort_values("quarterback_performance_grade", ascending=True)
)

fig = go.Figure()

# dumbbell lines
for _, row in pdf.iterrows():
    fig.add_trace(
        go.Scatter(
            x=[row["quarterback_support_grade"], row["quarterback_performance_grade"]],
            y=[row["player_short_name"], row["player_short_name"]],
            mode="lines",
            line=dict(
                color=row["team_color"],
                width=5,
            ),
            hoverinfo="skip",
            showlegend=False
        )
    )
    
fig.add_trace(
    go.Scatter(
        x=pdf["quarterback_support_grade"],
        y=pdf["player_short_name"],
        mode="markers",
        marker=dict(
            size=1, 
            opacity=0,
        ),
        customdata=pdf[
            ["team_abbr", "quarterback_support_grade"]
        ],
        hovertemplate=
        "<b>%{customdata[0]}</b><br>" +
        "Support Grade: %{customdata[1]:.1f}<extra></extra>",
        showlegend=False,
        hoverlabel=dict(
            bgcolor=pdf["team_color"],
            font_color="white"
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=pdf["quarterback_performance_grade"],
        y=pdf["player_short_name"],
        mode="markers",
        marker=dict(size=1, opacity=0),
        customdata=pdf[
            ["team_abbr", "quarterback_performance_grade"]
        ],
        hovertemplate=
        "<b>%{y}</b><br>" +
        "Performance Grade: %{customdata[1]:.1f}<extra></extra>",
        showlegend=False,
        hoverlabel=dict(
            bgcolor=pdf["team_color"],
            font_color="white"
        )
    )
)

for _, row in pdf.iterrows():
    fig.add_layout_image(
        dict(
            source=row["team_logo_espn"],
            x=row["quarterback_support_grade"],
            y=row["player_short_name"],
            xref="x",
            yref="y",
            sizex=2,
            sizey=0.9,
            xanchor="center",
            yanchor="middle",
        )
    )
    
    
    
for _, row in pdf.iterrows():
    fig.add_layout_image(
        dict(
            source=row["headshot"],
            x=row["quarterback_performance_grade"],
            y=row["player_short_name"],
            xref="x",
            yref="y",
            sizex=2,
            sizey=0.9,
            xanchor="center",
            yanchor="middle"
        )
    )

fig.update_layout(
    title=dict(text="Quarterback Support vs Performance (Top 25 QBs in attempts)", x=0.5),
    width=1500,
    height=1000, 
    legend_itemclick=False,
    xaxis=dict(
        title="Performance / Support Grade", 
    ),
    yaxis=dict(showticklabels=False)
)

fig.show()
fig.write_image("figures/images/qbs/qbs_support_vs_performance.png", scale=3)
fig.write_html("figures/html/qbs/qbs_support_vs_performance.html")

## Conclusions

**Key Takeaways**  
- **Support vs Performance** – Identify quarterbacks who over-perform relative to team support and those who rely heavily on strong support. *(Insert examples or specific player highlights)*  
- **Strengths & Weaknesses** – Summarize trends across the league in areas like pass protection, pass catching, play calling, and run game. *(Mention any standout teams or QBs)*  
- **High-Impact Metrics** – Highlight which metrics appear most predictive of performance or where differences between QBs are most pronounced. *(Could be EPA per dropback, completion over expected, etc.)*  

**Insights for Teams & Analysts**  
- **Roster Decisions** – Insights into which QBs might succeed with a change in supporting cast.  
- **Scheme Adjustments** – Where certain teams could improve play calling, protection, or receiving schemes to boost QB efficiency.  
- **Player Evaluation** – Identify QBs whose performance might be undervalued due to poor team support.  

**Next Steps / Further Analysis**  
- Compare across seasons to identify trends or development.  
- Include additional situational metrics (red zone, 3rd down efficiency, pressure split performance).  
- Explore interaction between support components and QB style (e.g., pocket passer vs. dual threat).  